In [ ]:
import torch
import model
import lancedb
import os
from hydra import initialize, initialize_config_module, initialize_config_dir, compose
from omegaconf import OmegaConf
from dotenv import load_dotenv
from datafusion import functions as f
from torch.utils.data import DataLoader, random_split
import torch.nn.functional as F
import numpy as np
from PIL import Image
from torchvision import transforms
import trackio

In [ ]:
with initialize(version_base=None, config_path="conf"):
    load_dotenv()
    write_url = os.environ["TRACKIO_WRITE_URL"]
    cfg = compose(config_name="config", overrides=[f'trackio.write_url="{write_url}"'])
    trackio.init(name=cfg.trackio.run_name, project=cfg.trackio.project, server_url=cfg.trackio.write_url)

In [ ]:
vae = model.AutoEncoder(cfg).to(cfg.device)

In [ ]:
class EpisodicDataset(torch.utils.data.Dataset):
    def __init__(self, cfg):
        super().__init__()
        self.db = lancedb.connect(cfg.dataset.lancedb_uri)
        self.table = self.db.open_table("episodes")

    def __getitem__(self, idx):
        episode = self.table.search().where(f'episode_id = {idx}').limit(1).to_arrow()
        obs_col = episode.column('observations').combine_chunks()  # ListArray, 1 row
        frames_py = obs_col[0].as_py()  # list of T lists, each 27648 uint8 ints
        frames_np = np.array(frames_py, dtype=np.uint8).reshape(-1, 96, 96, 3)
        t = torch.from_numpy(frames_np)  # (T, 96, 96, 3)
        return F.interpolate(
            t.permute(0, 3, 1, 2).to(torch.float32) / 255,
            (64, 64), mode="bilinear", align_corners=False
        )

    def __len__(self):
        return self.table.count_rows()

In [ ]:
ds = EpisodicDataset(cfg)
train_ds, valid_ds = random_split(ds, [0.9, 0.1])

In [ ]:
train_loader = DataLoader(train_ds, batch_size=1)
val_loader = DataLoader(valid_ds, batch_size=1)

In [ ]:
optimizer = torch.optim.AdamW(vae.parameters())
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, cfg.training.nb_epochs)

for _ in range(cfg.training.nb_epochs):
    for batch in train_loader:
        x = batch.squeeze(0).to(cfg.device)  # (T, 3, 64, 64)
        y = vae(x)
        loss = F.mse_loss(y, x)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        trackio.log({
            "training-loss": loss.item(),
        })
    with torch.no_grad():
        for batch in val_loader:
            x = batch.squeeze(0).to(cfg.device)  # (T, 3, 64, 64)
            loss = F.mse_loss(vae(x), x)
            trackio.log({
                "val-loss": loss.item()
            })
    scheduler.step()